# Dynamical.org STAC Catalog Builder

This notebook provides a fully self-contained workflow to:
1. **Discover** weather datasets from the [AWS Open Data Registry](https://registry.opendata.aws/) (looking for `dynamical.org` entries).
2. **Scan** public S3 buckets for Icechunk repositories.
3. **Extract** robust STAC metadata (including Datacube extensions via `xstac` and CRS-aware bounding boxes via `rioxarray`).
4. **Assemble** a static STAC catalog ready for publishing.

## Setup
Ensure you have the following installed:
```bash
pip install icechunk xarray zarr pystac xstac rioxarray s3fs requests pyyaml
```

In [ ]:
import os
import json
import warnings
import logging
import tempfile
import requests
import yaml
from datetime import datetime
from pathlib import Path
from typing import Any
from copy import deepcopy

import icechunk
import pystac
import rioxarray
import s3fs
import xarray as xr
import numpy as np

# Suppress Zarr V3 warnings
warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

logging.basicConfig(level=logging.INFO)

## 1. Metadata Extraction Logic
We include the core logic from `cloudify.stac.builder` here to make the notebook self-contained. This includes the robust BBox calculation and Datacube extension formatting.

In [ ]:
ICECHUNK_MEDIA_TYPE = "application/vnd.zarr+icechunk"
ICECHUNK_STAC_EXTENSIONS = [
    "https://stac-extensions.github.io/storage/v2.0.0/schema.json",
    "https://stac-extensions.github.io/virtual-assets/v1.0.0/schema.json",
    "https://stac-extensions.github.io/zarr/v1.1.0/schema.json",
    "https://stac-extensions.github.io/version/v1.2.0/schema.json",
    "https://stac-extensions.github.io/datacube/v2.2.0/schema.json",
]

def extract_spatial_extent_rio(ds: xr.Dataset) -> list[float]:
    """Return [lonmin, latmin, lonmax, latmax] in WGS84 using rioxarray."""
    try:
        first_var = next(iter(ds.data_vars))
        lonmin, latmin, lonmax, latmax = ds[first_var].rio.transform_bounds("EPSG:4326")
        # Clamp to avoid antimeridian artifacts and browser rejection
        return [
            max(-179.9, lonmin), max(-89.9, latmin), 
            min(179.9, lonmax), min(89.9, latmax)
        ]
    except Exception:
        return [-180.0, -90.0, 180.0, 90.0]

def build_datacube_extension(ds: xr.Dataset, time_min: str | None, time_max: str | None) -> dict:
    """Manual fallback for cube metadata if xstac is not desired for everything."""
    cube = {"cube:dimensions": {}, "cube:variables": {}}
    for dv in ds.data_vars:
        cube["cube:variables"][dv] = {
            "type": "data",
            "dimensions": list(ds[dv].dims),
            "unit": ds[dv].attrs.get("units", "not set"),
            "description": ds[dv].attrs.get("long_name", str(dv)),
        }
    if time_min and time_max:
        cube["cube:dimensions"]["time"] = {"type": "temporal", "extent": [time_min, time_max]}
    return cube

def _bbox_to_geometry(bbox: list[float]) -> dict:
    l, b, r, t = bbox
    return {"type": "Polygon", "coordinates": [[[l, b], [l, t], [r, t], [r, b], [l, b]]]}

def make_json_serializable(obj: Any) -> Any:
    if isinstance(obj, dict): return {k: make_json_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [make_json_serializable(v) for v in obj]
    if isinstance(obj, (datetime, np.datetime64)): return str(obj)
    return obj

## 2. Discovery Logic
These functions query the GitHub-hosted Open Data Registry and scan S3 for `.icechunk` prefixes.

In [ ]:
REGISTRY_API = "https://api.github.com/repos/awslabs/open-data-registry/contents/datasets?per_page=300"
REGISTRY_RAW = "https://raw.githubusercontent.com/awslabs/open-data-registry/main/datasets/{filename}"

def fetch_registry_entries():
    print("Querying AWS Open Data Registry for dynamical.org datasets...")
    resp = requests.get(REGISTRY_API, timeout=30)
    resp.raise_for_status()
    files = [f for f in resp.json() if f["name"].startswith("dynamical-") and f["name"].endswith(".yaml")]
    
    entries = []
    for f in files:
        raw = requests.get(REGISTRY_RAW.format(filename=f["name"]), timeout=30)
        entry = yaml.safe_load(raw.text)
        entry["_filename"] = f["name"]
        entries.append(entry)
    return entries

def discover_icechunk_prefixes(bucket: str, region: str):
    fs = s3fs.S3FileSystem(anon=True, client_kwargs={"region_name": region})
    prefixes = []
    try:
        top_paths = fs.ls(bucket, detail=False)
        for top_path in top_paths:
            sub_paths = fs.ls(top_path, detail=False)
            for sub_path in sub_paths:
                if sub_path.endswith(".icechunk"):
                    prefix = sub_path[len(bucket) + 1:].rstrip("/") + "/"
                    prefixes.append(prefix)
    except Exception as e:
        print(f"  Error scanning bucket {bucket}: {e}")
    return prefixes

def bucket_from_entry(entry: dict):
    for resource in entry.get("Resources", []):
        if resource.get("Type") == "S3 Bucket":
            bucket = resource.get("ARN", "").split(":::")[-1]
            region = resource.get("Region", "us-east-1")
            return bucket, region
    return None, None

## 3. Discovery and Item Building
Now we run the discovery loop. We open each store briefly to extract the snapshot ID and coordinate ranges.

In [ ]:
catalog = pystac.Catalog(
    id="dynamical-org-discovery",
    description="Dynamically discovered Icechunk stores from dynamical.org on AWS Open Data.",
    catalog_type=pystac.CatalogType.SELF_CONTAINED,
)

entries = fetch_registry_entries()
DYNAMICAL_PROVIDER = pystac.Provider(name="dynamical.org", roles=["producer", "processor", "host"], url="https://dynamical.org")

for entry in entries:
    bucket, region = bucket_from_entry(entry)
    if not bucket:
        continue
        
    print(f"\nProcessing {entry['Name']} (s3://{bucket})...")
    prefixes = discover_icechunk_prefixes(bucket, region)
    
    for prefix in prefixes:
        store_uri = f"s3://{bucket}/{prefix}"
        print(f"  Found store: {store_uri}")
        
        try:
            storage = icechunk.s3_storage(bucket=bucket, prefix=prefix, region=region, anonymous=True)
            repo = icechunk.Repository.open(storage=storage)
            session = repo.readonly_session(branch="main")
            ds = xr.open_zarr(session.store, chunks=None, consolidated=False, zarr_format=3)
            
            # 1. BBox via rioxarray
            bbox = extract_spatial_extent_rio(ds)
            
            # 2. Dimensions & CRS
            time_dim = next((d for d in ["time", "init_time"] if d in ds.dims), None)
            x_dim = next((d for d in ["lon", "longitude", "x"] if d in ds.dims), None)
            y_dim = next((d for d in ["lat", "latitude", "y"] if d in ds.dims), None)
            
            try:
                ref_sys = ds.rio.crs.to_epsg() or ds.rio.crs.to_wkt()
            except Exception:
                ref_sys = 4326

            start_dt = end_dt = None
            if time_dim:
                start_dt = datetime.fromisoformat(str(ds[time_dim].min().values[()]).split(".")[0])
                end_dt = datetime.fromisoformat(str(ds[time_dim].max().values[()]).split(".")[0])
            
            # 3. Create Item
            dataset_name = prefix.split("/")[0]
            version_str = prefix.split("/")[1]
            item_id = f"{dataset_name}-{version_str.replace('.icechunk', '').replace('.', '-')}"

            # Prettier title including version and sub-product
            base_name = entry.get("Name", dataset_name).split("(")[0].strip()
            sub_prod = dataset_name.replace("noaa-", "").replace("nasa-", "").replace("-", " ").title()
            for word in base_name.split():
                sub_prod = sub_prod.replace(word, "").strip()
            
            clean_version = version_str.replace(".icechunk", "")
            item_title = f"{base_name} ({sub_prod.title()}, {clean_version})"

            item = pystac.Item(
                id=item_id,
                geometry=_bbox_to_geometry(bbox),
                bbox=bbox,
                datetime=None,
                start_datetime=start_dt,
                end_datetime=end_dt,
                properties={"title": item_title, "description": entry.get("Description", "")},
                stac_extensions=ICECHUNK_STAC_EXTENSIONS,
            )
            
            # 4. Storage Schemes (required for xpystac)
            item.extra_fields["storage:schemes"] = {
                f"aws-s3-{bucket}": {"type": "aws-s3", "bucket": bucket, "region": region, "anonymous": True}
            }
            
            # 5. Add Primary Asset
            item.add_asset(
                f"data@{session.snapshot_id}",
                pystac.Asset(
                    href=store_uri,
                    media_type=ICECHUNK_MEDIA_TYPE,
                    roles=["data", "references"],
                    extra_fields={
                        "icechunk:snapshot_id": session.snapshot_id,
                        "version": session.snapshot_id,
                        "storage:refs": [f"aws-s3-{bucket}"]
                    }
                )
            )
            
            # 6. Datacube Extension via xstac (if available)
            try:
                import xstac
                item = xstac.xarray_to_stac(
                    ds, 
                    item, 
                    temporal_dimension=time_dim, 
                    x_dimension=x_dim, 
                    y_dimension=y_dim, 
                    reference_system=ref_sys,
                    validate=False
                )
            except ImportError:
                cube = build_datacube_extension(ds, start_dt.isoformat() + "Z" if start_dt else None, end_dt.isoformat() + "Z" if end_dt else None)
                item.properties.update(cube)
                
            catalog.add_item(item)
            print(f"    Successfully added Item: {item_id}")
            
        except Exception as e:
            print(f"    Failed to build item for {store_uri}: {e}")

## 4. Save and Review
We save the catalog to a local temporary directory.

In [ ]:
output_dir = Path("./stac_out")
output_dir.mkdir(exist_ok=True)

catalog.normalize_hrefs(str(output_dir))
catalog.save()

print(f"\nCatalog saved to: {output_dir.absolute()}")
print(f"Total items: {len(list(catalog.get_items()))}")

# Show the top-level catalog.json
with open(output_dir / "catalog.json") as f:
    print(json.dumps(json.load(f), indent=2))